# 🏗️ 38 · Protein structure: geometry, contacts and structure prediction
### Cα geometry and helix handedness, superposition with the Kabsch algorithm, RMSD, TM-score and lDDT, distance geometry with classical MDS, folding a protein from coevolution contacts, confidence from ensembles, and how AlphaFold works

**BiologyToML** · Level 2 · Research frontier · Part H — Genomes, proteins and molecules · Module 38 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/38_protein_structure.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, PyTorch (CPU), pandas, Matplotlib |
| **Prerequisites** | Level 1 (especially Modules 03, 04 and 14) and [Module 37](37_protein_sequences_language_models.ipynb) (alignments, Potts models, direct-coupling analysis); [Module 25](25_graph_neural_networks.ipynb) (equivariance) is helpful |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [03 · Linear algebra](03_linear_algebra_for_data.ipynb), [04 · Calculus and optimisation](04_calculus_and_optimization.ipynb), [14 · PCA](14_pca_dimensionality_reduction.ipynb), [21 · Bayesian learning and MCMC](21_bayesian_learning_mcmc.ipynb), [24 · Attention and transformers](24_attention_transformers.ipynb), [28 · Diffusion and flow matching](28_diffusion_flow_matching.ipynb), [25 · Graph neural networks](25_graph_neural_networks.ipynb), [37 · Protein sequences](37_protein_sequences_language_models.ipynb) |
| **Research frontier covered** | protein structure representation and comparison (RMSD, TM-score, lDDT), distance geometry, contact-guided folding, AlphaFold 2/3 and RoseTTAFold, confidence estimation — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All structures and sequences are **SIMULATED**: the Cα trace of an idealised 68-residue four-helix bundle
> (`simulate_helix_bundle`, which we check against physics in Section 2: every helix is right-handed) and a protein
> family of 1,000 aligned sequences drawn by `simulate_protein_family` from a Potts model
> whose couplings sit exactly on that structure's contacts. The true coordinates, distances and contacts are known
> exactly, so every prediction can be scored. Real proteins have all atoms, β-sheets, irregular loops, ligands and
> experimental errors, and real alignments have gaps and phylogeny — see 🧭 *From toy to real data*.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Describe** how a protein structure is represented (backbone, Cα trace, secondary structure, PDB entries) and how X-ray crystallography, NMR and cryo-EM determine it.
2. **Compute** Cα virtual bond lengths, virtual angles, pseudo-torsions, distance maps and contact maps, **recognise** the α-helix signature and its handedness — and **use** it to catch an error in a simulator.
3. **Derive and implement** the Kabsch superposition algorithm with the SVD (including the reflection case) and **compute** RMSD, TM-score and lDDT, **explaining** which perturbations each one forgives.
4. **Reconstruct** coordinates from a distance matrix with classical MDS, **explain** why the answer is only defined up to a mirror image, and **embed** noisy, incomplete distances by gradient descent.
5. **Fold** a protein from contacts predicted by direct-coupling analysis plus secondary structure, and **evaluate** the models against the truth for different alignment depths and numbers of contacts.
6. **Estimate** per-residue confidence from an ensemble of models (a pLDDT-like score) and **test** whether it predicts the real error.
7. **Explain** the architecture of AlphaFold 2 (Evoformer, triangle updates, invariant point attention, recycling, FAPE, pLDDT and PAE), what AlphaFold 3, RoseTTAFold and ESMFold change, and **compute** an aligned-error matrix.
8. **Identify** what current structure predictors do not do well, and the first things that go wrong with real data.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · 🔬 Protein structure primer | backbone, Cα trace, secondary structure; experimental methods; the toy protein | 15 min |
| 2 · Geometry | virtual bonds, angles, pseudo-torsions; 🧪 a simulator check; distance and contact maps | 20 min |
| 3 · Comparing structures | rotations; Kabsch; RMSD, TM-score, lDDT; 🧪 a hinge motion | 30 min |
| 4 · From distances to coordinates | classical MDS and the mirror image; noisy and incomplete distances | 25 min |
| 5 · From predicted contacts to 3-D | mean-field DCA; restraint folding; 🧪 TM-score vs depth and contact number | 30 min |
| 6 · Confidence | ensembles; a pLDDT-like score; 🧪 does confidence predict error? | 15 min |
| 7 · How AlphaFold works | Evoformer, triangle updates, IPA, recycling, pLDDT/PAE; AF3, RoseTTAFold, ESMFold | 20 min |
| 8 · 🧭 From toy to real data | PDB, AlphaFold DB, ColabFold, Foldseek, CASP; what goes wrong first | 10 min |
| ✅ / 🧠 / 🚀 / 📝 | takeaways, checkpoint, discussion, going further, lab note | 10 min |

## 🧫 The biological question

A protein is synthesised as a chain of amino acids, yet it works as a precisely folded three-dimensional object: an
enzyme's active site, a receptor's binding pocket and an antibody's grip on its antigen are all matters of shape. In the
1960s Anfinsen showed that a small protein refolds into its active shape on its own, so that — at least for such
proteins — **the sequence determines the structure** (Anfinsen 1973). Determining structures experimentally remained slow
and expensive: by 2020 the Protein Data Bank held fewer than 200,000 experimental structures, against hundreds of millions
of known sequences.

In the CASP14 blind assessment of 2020, AlphaFold 2 predicted many structures with accuracy competitive with experiment
(Jumper *et al.* 2021), and its database of predicted structures (Varadi *et al.* 2022) has since grown to more than 200
million entries, covering most known protein sequences. To use these predictions well — and to understand what they cannot do — a biologist needs
answers to three questions:

1. **What exactly is predicted, and how is a prediction compared with an experimental structure?** Two structures in
   different orientations cannot be compared atom by atom until they are superposed; and a single number (RMSD,
   TM-score, lDDT) must summarise how similar they are.
2. **How can sequence information become coordinates?** The path that led to AlphaFold went through *contacts*: pairs of
   residues that touch, predicted from coevolution in a family alignment (Module 37), then turned into 3-D coordinates by
   distance geometry and optimisation (Marks *et al.* 2011).
3. **How confident should we be?** AlphaFold reports a per-residue confidence (pLDDT) and a pairwise one (PAE); what do
   they measure?

This notebook builds each piece from scratch on a SIMULATED four-helix bundle — a common fold of small all-α domains and
cytokines — whose true structure is known exactly. It is the same protein whose contact map generated the families of
Modules 24 and 37, so the whole chain *family → contacts → structure → confidence* can be checked against the truth.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from scipy.sparse.csgraph import shortest_path
from scipy.stats import spearmanr

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration and colours ──
torch.manual_seed(SEED)
torch.set_num_threads(1)                 # tiny optimisations: one thread is as fast and avoids slow-downs on shared CPUs
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#E69F00", "#CC79A7", "#56B4E9", "#F0E442"]   # colour-blind-safe
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads())

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_helix_bundle` returns the Cα coordinates, secondary structure, distance map and contact map of an idealised four-helix bundle; `simulate_protein_family` samples an alignment of a protein family from a Potts model whose couplings sit on a given contact map (as in Module 37). Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np

def simulate_helix_bundle(n_helices: int = 4, helix_length: int = 14, loop_length: int = 4,
                          rng=None, noise: float = 0.0) -> dict:
    """Cα coordinates (Å) of an idealised four-helix-bundle protein, its contact map and secondary structure.

    Helices are ideal right-handed α-helices (3.6 residues per turn, 1.5 Å rise, 2.3 Å radius)
    packed antiparallel on the corners of a 10 Å square and joined by short loops.  ``noise`` adds
    Gaussian jitter (needs ``rng``).  Contacts: Cα–Cα distance < 8 Å and |i − j| ≥ 3.

    Returns a dict with ``coords`` (L × 3), ``ss`` (string of 'H'/'L'), ``distances`` and ``contacts``.
    """
    corners = 5.0 * np.array([[1, 1], [1, -1], [-1, -1], [-1, 1]], float)
    pts, ss = [], ""
    for h in range(n_helices):
        cx, cy = corners[h % 4] * (1 + h // 4)
        up = h % 2 == 0
        k = np.arange(helix_length)
        angle = np.deg2rad(100.0) * k * (1 if up else -1)      # turn with the direction of travel, so
        z = 1.5 * (k - (helix_length - 1) / 2) * (1 if up else -1)   # every helix is right-handed
        helix = np.c_[cx + 2.3 * np.cos(angle), cy + 2.3 * np.sin(angle), z]
        if pts:                                                      # loop: circular arc, 3.8 Å steps
            start, end = pts[-1], helix[0]
            d, m = np.linalg.norm(end - start), loop_length + 1
            lo, hi = 1e-6, 2 * np.pi - 1e-6                              # arc angle Θ with m equal chords
            for _ in range(60):
                theta = (lo + hi) / 2
                lo, hi = (theta, hi) if 3.8 * np.sin(theta / 2) / np.sin(theta / (2 * m)) > d else (lo, theta)
            R = 1.9 / np.sin(theta / (2 * m))
            e1 = (end - start) / d
            e2 = np.r_[0, 0, np.sign(start[2])] - np.dot(np.r_[0, 0, np.sign(start[2])], e1) * e1
            e2 /= np.linalg.norm(e2)
            c = (start + end) / 2 - R * np.cos(theta / 2) * e2
            for k in range(1, m):
                psi = -theta / 2 + k * theta / m
                pts.append(c + R * (np.sin(psi) * e1 + np.cos(psi) * e2))
            ss += "L" * loop_length
        pts.extend(helix)
        ss += "H" * helix_length
    coords = np.array(pts)
    if noise > 0:
        coords = coords + rng.normal(0, noise, coords.shape)
    D = np.linalg.norm(coords[:, None] - coords[None], axis=-1)
    sep = np.abs(np.arange(len(coords))[:, None] - np.arange(len(coords))[None])
    return dict(coords=coords, ss=ss, distances=D, contacts=(D < 8.0) & (sep >= 3))


def potts_score(seqs, h, J) -> np.ndarray:
    """Statistical fitness Σ_i h_i(a_i) + Σ_{i<j} J_ij(a_i, a_j) of integer-coded sequences (higher = fitter)."""
    seqs = np.atleast_2d(seqs)
    L = seqs.shape[1]
    ii, jj = np.nonzero(np.triu(np.abs(J).sum(axis=(2, 3)) > 0, 1))      # coupled pairs only
    return h[np.arange(L), seqs].sum(axis=1) + J[ii, jj, seqs[:, ii], seqs[:, jj]].sum(axis=1)


def simulate_protein_family(rng: np.random.Generator, contacts: np.ndarray, n_seqs: int = 3000,
                            q: int = 20, field_scale: float = 1.0, coupling_scale: float = 1.0,
                            n_sweeps: int = 100) -> dict:
    """Simulate a protein family (a multiple sequence alignment) from a Potts model on a contact map.

    Each residue pair in contact gets a q × q coupling matrix J_ij (pairs that fit together
    are rewarded), each position a field h_i (its conservation).  Sequences are drawn by
    Gibbs sampling (``n_seqs`` independent chains, ``n_sweeps`` sweeps each), so conserved
    positions and *co-varying* contact pairs appear exactly as in natural alignments — but
    without phylogeny.  The wild type is the fittest sampled sequence.

    Returns a dict with ``msa`` (n_seqs × L integers 0…q−1; letters = AMINO_ACIDS[:q]),
    ``h`` (L × q), ``J`` (L × L × q × q, symmetric: J[j, i] = J[i, j].T), ``wild_type``.
    """
    L = contacts.shape[0]
    cons = rng.uniform(0.3, 1.5, L)[:, None] * field_scale
    h = rng.normal(0, 1, (L, q)) * cons
    J = np.zeros((L, L, q, q))
    pairs = [(i, j) for i in range(L) for j in range(i + 1, L) if contacts[i, j]]
    for i, j in pairs:
        J[i, j] = rng.normal(0, 0.5 * coupling_scale, (q, q))
        J[j, i] = J[i, j].T
    nbrs = [np.flatnonzero(contacts[i]) for i in range(L)]
    x = rng.integers(0, q, (n_seqs, L))
    rows = np.arange(n_seqs)
    for _ in range(n_sweeps):
        for i in rng.permutation(L):
            logits = np.tile(h[i], (n_seqs, 1))
            for j in nbrs[i]:
                logits += J[j, i][x[:, j]]                                 # = J_ij(·, x_j)
            g = logits - np.log(-np.log(rng.random((n_seqs, q))))           # Gumbel-max sampling
            x[rows, i] = np.argmax(g, axis=1)
    wild_type = x[np.argmax(potts_score(x, h, J))].copy()
    return dict(msa=x, h=h, J=J, wild_type=wild_type)


print("🧰 Toolkit loaded:", ", ".join(['simulate_helix_bundle', 'potts_score', 'simulate_protein_family']))

---
## 1 · 🔬 Protein structure primer

> 📍 **Where we are.** Modules 24 and 37 treated a protein as a *sequence*, with a contact map in the background. Here
> the 3-D structure takes centre stage: what it is, how it is measured, and how it is stored.

**From chain to fold.** Every amino acid contributes the same backbone atoms — N, Cα, C (and O) — plus a side chain.
Consecutive residues are joined by a planar peptide bond, which fixes the distance between consecutive Cα atoms at
**3.8 Å** (1 Å = 0.1 nm). The backbone can rotate only about two bonds per residue (the dihedral angles φ and ψ), and
steric clashes restrict those to a few allowed regions (the Ramachandran plot; Ramachandran, Ramakrishnan & Sasisekharan
1963). Two regular patterns dominate: the **α-helix** (3.6 residues per turn, 1.5 Å rise per residue, held by hydrogen
bonds from residue $i$ to $i+4$; Pauling, Corey & Branson 1951) and the **β-strand**, which pairs with other strands into
sheets. Helices, strands and the loops between them pack into a compact **tertiary structure**.

A useful simplification — used throughout this notebook — is the **Cα trace**: one point per residue at its Cα atom.
It keeps the fold and loses the side chains.

🔬 **Assay box — how structures are determined.**

| Method | What is measured | Typical output | Typical artefacts and limits |
|---|---|---|---|
| X-ray crystallography | diffraction of X-rays by a protein crystal | electron density → atomic model, resolution 1–3 Å | needs crystals; crystal packing; flexible regions missing; one static conformation |
| NMR spectroscopy | magnetic interactions between nuclei, e.g. distances < 5–6 Å between hydrogens (NOEs) | an *ensemble* of models fitted to distance restraints | small proteins (≲ 30–40 kDa); restraints are sparse and imprecise |
| Cryo-electron microscopy | images of frozen single molecules (or cells, by tomography) | 3-D density map → model, often 2–4 Å for large complexes | small proteins hard; preferred orientations; heterogeneity |

All three deposit atomic models in the **Protein Data Bank** (PDB; Berman *et al.* 2000): a file lists, for each atom,
its residue, chain and $x,y,z$ coordinates in Å, with a per-atom uncertainty (B-factor) and the experimental metadata.
NMR's distance restraints are the experimental cousin of Section 4's distance geometry, and the sister course
[BiologyToResearch](../../../../cryoem/tutorial/BiologyToResearch/) teaches cryo-EM from image formation to atomic models.

**Our protein.** `simulate_helix_bundle` builds a 68-residue protein of four ideal α-helices (14 residues each) on the
corners of a 10 Å square, joined by three 4-residue loops — an idealised **four-helix bundle**.

In [ ]:
# ── 1.1 The SIMULATED protein: an idealised four-helix bundle ──
bundle = simulate_helix_bundle()
X_true, ss, D_true, contacts = bundle["coords"], bundle["ss"], bundle["distances"], bundle["contacts"]
L = len(X_true)
SEP = np.abs(np.arange(L)[:, None] - np.arange(L)[None])          # sequence separation |i − j|
LONG = contacts & (SEP >= 6)                                        # long-range contacts: not helix neighbours
IS_HELIX = np.array([c == "H" for c in ss])
SEGMENT = np.cumsum(np.r_[0, np.diff(IS_HELIX.astype(int)) != 0])          # 0, 1, 2, … for helix/loop segments
HELICES = [np.flatnonzero((SEGMENT == s) & IS_HELIX) for s in np.unique(SEGMENT[IS_HELIX])]
print(f"SIMULATED protein: {L} residues; secondary structure {ss}")
print(f"{len(HELICES)} helices: residues " + ", ".join(f"{h[0]}–{h[-1]}" for h in HELICES))
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot(*X_true.T, color="0.6", lw=1)
ax.scatter(*X_true.T, c=[OKABE_ITO[0] if h else OKABE_ITO[1] for h in IS_HELIX], s=14, depthshade=False)
ax.set_xlabel("x (Å)"); ax.set_ylabel("y (Å)"); ax.set_zlabel("z (Å)")
ax.set_title("Cα trace (blue = helix, orange = loop)")
ax2 = fig.add_subplot(1, 2, 2)
for k, h in enumerate(HELICES):
    ax2.plot(*X_true[h, :2].T, "o-", ms=3, lw=0.8, color=OKABE_ITO[k], label=f"helix {k + 1}")
    ax2.annotate("", xy=X_true[h[-1], :2], xytext=X_true[h[-2], :2], arrowprops=dict(arrowstyle="->", color=OKABE_ITO[k]))
ax2.set_aspect("equal"); ax2.set_xlabel("x (Å)"); ax2.set_ylabel("y (Å)"); ax2.legend(fontsize=7, loc="center")
ax2.set_title("view down the bundle axis: four helices on a 10 Å square")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The trace shows four helices running alternately up and down (an *up–down–up–down* bundle),
joined by short loops. Seen from above, each helix projects onto a circle of radius 2.3 Å around one corner of a 10 Å
square. Real four-helix bundles have the same overall architecture but irregular, side-chain-packed helices.

✏️ **Your turn.** Call `simulate_helix_bundle(n_helices=3)` and `simulate_helix_bundle(n_helices=5)` and plot their
top views. Where does the fifth helix go?

---
## 2 · Geometry: describing a fold with numbers

> 📍 **Where we are.** Coordinates depend on where the protein sits and how it is turned. Quantities that do *not* —
> distances, angles and dihedral angles between atoms — describe the fold itself. They are also the first thing to check
> in any structure, measured or simulated.

🧮 **Math refresher — angles and dihedral (torsion) angles.** For three consecutive Cα atoms $a,b,c$ the **virtual bond
angle** at $b$ is

$$\theta_b=\arccos\frac{(a-b)\cdot(c-b)}{\lVert a-b\rVert\,\lVert c-b\rVert}.$$

*In words:* the angle between the two bonds that meet at $b$, from the dot product of [Module 03](03_linear_algebra_for_data.ipynb).
For four consecutive atoms $a,b,c,d$ with bond vectors $b_1=b-a$, $b_2=c-b$, $b_3=d-c$, the **pseudo-torsion** (dihedral
angle) is the angle between the plane through $a,b,c$ and the plane through $b,c,d$, measured around the middle bond:

$$\tau=\operatorname{atan2}\!\Big(\lVert b_2\rVert\,b_1\cdot(b_2\times b_3),\;(b_1\times b_2)\cdot(b_2\times b_3)\Big).$$

*In words:* the cross products $b_1\times b_2$ and $b_2\times b_3$ are perpendicular to the two planes; the second argument
measures how aligned the planes are, the first on which side $d$ lies. Unlike the bond angle, the torsion has a **sign**:
a mirror image flips it. That makes it the natural measure of **handedness**. In a right-handed α-helix the Cα
pseudo-torsion is about $+50°$ (and the virtual bond angle about $90°$); a left-handed helix would give about $-50°$;
an extended β-strand gives angles near $120°$ and torsions near $\pm180°$.

In [ ]:
# ── 2.1 Virtual bonds, virtual angles and pseudo-torsions of the Cα trace ──
def virtual_bonds(X):
    """Distances between consecutive Cα atoms (L − 1 values)."""
    return np.linalg.norm(np.diff(X, axis=0), axis=1)


def virtual_angles(X):
    """Angle (degrees) at every inner Cα between its two virtual bonds (L − 2 values, residues 1 … L − 2)."""
    a, b = X[:-2] - X[1:-1], X[2:] - X[1:-1]
    cos = np.sum(a * b, axis=1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


def pseudo_torsions(X):
    """Signed dihedral angle (degrees, IUPAC convention) of every 4 consecutive Cα atoms (L − 3 values)."""
    b1, b2, b3 = X[1:-2] - X[:-3], X[2:-1] - X[1:-2], X[3:] - X[2:-1]
    y = np.linalg.norm(b2, axis=1) * np.sum(b1 * np.cross(b2, b3), axis=1)
    x = np.sum(np.cross(b1, b2) * np.cross(b2, b3), axis=1)
    return np.degrees(np.arctan2(y, x))


def helix_torsions(X):
    """Mean pseudo-torsion of each helix, using only quadruples of atoms that lie inside the helix."""
    tau = pseudo_torsions(X)
    return np.array([tau[h[0]:h[-1] - 2].mean() for h in HELICES])


print(f"virtual bonds : {virtual_bonds(X_true).min():.2f}–{virtual_bonds(X_true).max():.2f} Å")
print(f"virtual angles inside helices: {np.round(np.unique(virtual_angles(X_true)[IS_HELIX[1:-1] & IS_HELIX[:-2] & IS_HELIX[2:]].round()), 0)} degrees")
print("mean pseudo-torsion of helices 1–4 (degrees):", np.round(helix_torsions(X_true), 1))
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharex=True)
for ax, (vals, offset, lab) in zip(axes, [(virtual_bonds(X_true), 0.5, "virtual bond (Å)"),
                                          (virtual_angles(X_true), 1, "virtual angle (°)"),
                                          (pseudo_torsions(X_true), 1.5, "pseudo-torsion (°)")]):
    for h in HELICES:
        ax.axvspan(h[0], h[-1], color=OKABE_ITO[5], alpha=0.15, lw=0)
    ax.plot(np.arange(len(vals)) + offset, vals, ".-", color="0.2", lw=0.8, ms=3)
    ax.set_xlabel("residue (shaded = helix)"); ax.set_ylabel(lab)
axes[2].axhline(50, color=OKABE_ITO[2], ls="--", lw=1); axes[2].axhline(-50, color=OKABE_ITO[1], ls="--", lw=1)
axes[0].set_title("SIMULATED bundle: 3.8 Å everywhere"); axes[1].set_title("≈ 90° inside helices")
axes[2].set_title("+50° in every helix: right-handed")
plt.tight_layout(); plt.show()

🔍 **What just happened? — 🧪 a ground-truth check of the simulator itself.** Before using simulated structures we check
them against known physics, exactly as structure-validation software checks bond geometry, Ramachandran angles and
chirality for every PDB deposition. The check passes: virtual bonds are 3.80–3.83 Å, virtual angles inside helices are
90°, and all four helices have pseudo-torsion $+50°$ — **right-handed**, as α-helices built from L-amino acids are. This
is not a formality. A helix whose $z$ coordinate runs *down* must also turn the other way to stay right-handed; a
generator that forgets this produces left-handed "down" helices while its distances, angles and — almost — its contact
map look perfectly normal (an early version of this course's simulator did exactly that, and only this torsion check
caught it). **Validate a simulator against known physics before trusting what it teaches you.**

What would a wrong hand look like? The cleanest example is the **mirror image** of the whole protein.

In [ ]:
# ── 2.2 The mirror image: identical distances, opposite handedness ──
X_mirror = X_true * np.array([1, 1, -1])                               # reflect through the xy-plane (z → −z)
D_mirror = np.linalg.norm(X_mirror[:, None] - X_mirror[None], axis=-1)
print(f"largest change in any virtual bond   : {np.abs(virtual_bonds(X_mirror) - virtual_bonds(X_true)).max():.1e} Å")
print(f"largest change in any virtual angle  : {np.abs(virtual_angles(X_mirror) - virtual_angles(X_true)).max():.1e}°")
print(f"largest change in any distance       : {np.abs(D_mirror - D_true).max():.1e} Å  → identical contact maps: "
      f"{np.array_equal(D_mirror < 8, D_true < 8)}")
print("mean pseudo-torsion of helices 1–4   :", np.round(helix_torsions(X_true), 1), "(protein) →",
      np.round(helix_torsions(X_mirror), 1), "(mirror image)")
fig, ax = plt.subplots(figsize=(10, 2.8))
for h in HELICES:
    ax.axvspan(h[0], h[-1], color=OKABE_ITO[5], alpha=0.15, lw=0)
ax.plot(np.arange(L - 3) + 1.5, pseudo_torsions(X_true), ".-", color=OKABE_ITO[2], lw=0.8, ms=3, label="protein (right-handed)")
ax.plot(np.arange(L - 3) + 1.5, pseudo_torsions(X_mirror), ".-", color=OKABE_ITO[1], lw=0.8, ms=3, label="mirror image (left-handed)")
ax.set_xlabel("residue (shaded = helix)"); ax.set_ylabel("pseudo-torsion (°)")
ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1.01, 0.5))
ax.set_title("Only the signed torsions tell a protein from its mirror image")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Reflecting the protein changes **no** bond length, angle or distance (all differences are exactly
zero), so its distance map and contact map are identical — yet every helix turns from $+50°$ to $-50°$. The
mirror image is a different molecule: built from D-amino acids it could exist, but no natural protein looks like it. This
matters throughout the notebook. Contact maps, distance maps and every distance-based score are blind to the
difference; Section 3 shows that the right superposition is not, and Sections 4–5 use exactly this torsion check to
decide whether a structure reconstructed from distances came out the right way round.

In [ ]:
# ── 2.3 Distance map and contact map of the protein ──
print(f"closest non-neighbour Cα pair: {D_true[SEP >= 2].min():.2f} Å; contacts (Cα–Cα < 8 Å, |i − j| ≥ 3): "
      f"{contacts.sum() // 2} pairs, of which {LONG.sum() // 2} long-range (|i − j| ≥ 6)")
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
im = axes[0].imshow(D_true, cmap="viridis_r")
plt.colorbar(im, ax=axes[0], shrink=0.8, label="Cα–Cα distance (Å)")
axes[0].set_title("distance map")
axes[1].imshow(contacts & (SEP < 6), cmap="Greys", alpha=0.5)
ii, jj = np.nonzero(np.triu(LONG))
axes[1].scatter(jj, ii, s=10, color=OKABE_ITO[1], label="long-range contact (|i − j| ≥ 6)")
axes[1].scatter(ii, jj, s=10, color=OKABE_ITO[1])
axes[1].set_title("contact map (grey: local contacts inside helices)"); axes[1].legend(fontsize=7, loc="lower left")
for ax in axes:
    for h in HELICES[1:]:
        ax.axhline(h[0] - 2.5, color="w" if ax is axes[0] else "0.7", lw=0.5)
        ax.axvline(h[0] - 2.5, color="w" if ax is axes[0] else "0.7", lw=0.5)
    ax.set_xlabel("residue j"); ax.set_ylabel("residue i")
plt.tight_layout(); plt.show()

🔍 **What just happened?** No two non-neighbouring Cα atoms are closer than 5 Å (no clashes), and 128 residue pairs are in
contact. Read the maps like a biologist:

* **Bands along the diagonal**: inside a helix, residues $i$ and $i+3$, $i+4$ are close (5.0 and 6.2 Å) — the helical
  signature in a contact map.
* **Off-diagonal stripes perpendicular to the diagonal**: antiparallel neighbouring helices (1–2, 2–3, 3–4, and 1–4
  across the square). Parallel helices would give stripes parallel to the diagonal.
* No contacts between helices 1 and 3 or 2 and 4: they sit on opposite corners, 14 Å apart.

Only 38 pairs are long-range contacts — about $0.56L$ for $L=68$. Real proteins, judged by the usual $C_\beta$–$C_\beta$
< 8 Å definition, typically have more (often more than $L$), because side chains pack more tightly than our idealised
Cα model.

⚠️ **Pitfall.** Contact definitions differ between papers (Cα or Cβ atoms, 8 Å or other cut-offs, minimum sequence
separation 6, 12 or 24). CASP's assessment of contact prediction uses Cβ atoms and calls $|i-j|\ge24$ "long-range". Always
check the definition before comparing numbers.

✏️ **Your turn.** Build a structure in which only helix 2 is mirrored: reflect residues 18–31 through the plane that
contains the helix axis (e.g. $y\to2c_y-y$ with $c_y$ the mean $y$ of those residues). Which numbers of 2.1 and 2.3
reveal the error, and how many contacts change?

---
## 3 · Comparing structures: superposition, RMSD, TM-score and lDDT

> 📍 **Where we are.** To say how good a predicted structure is, we compare it with the experimental one. Their
> coordinates live in different frames, so we must first find the rigid motion that best overlays them.

**RMSD.** For two structures with matched atoms $p_i$ and $q_i$ ($i=1..N$), the root-mean-square deviation is

$$\mathrm{RMSD}=\sqrt{\frac1N\sum_{i=1}^N\lVert p_i-q_i\rVert^2}.$$

*In words:* the typical distance between corresponding atoms, in Å. It is meaningful only after **superposition** — the
rotation $R$ and translation $t$ that minimise $\sum_i\lVert Rp_i+t-q_i\rVert^2$.

🧮 **Math refresher — rotation matrices.** A rotation in 3-D is a $3\times3$ matrix $R$ with orthonormal columns
($R^\top R=I$, so lengths and angles are preserved) and $\det R=+1$. An orthogonal matrix with $\det R=-1$ is a
**reflection** (combined with a rotation): it preserves all distances but turns a right hand into a left hand.

**The Kabsch algorithm (Kabsch 1976).** The best translation moves the centroid of $P$ onto that of $Q$, so centre both
structures first. Then

$$\sum_i\lVert Rp_i-q_i\rVert^2=\sum_i\lVert p_i\rVert^2+\sum_i\lVert q_i\rVert^2-2\,\mathrm{tr}(RH),\qquad H=\sum_i p_i\,q_i^\top .$$

*In words:* only the last term depends on $R$, so we must *maximise* $\mathrm{tr}(RH)$, where the $3\times3$ matrix $H$ is
the cross-covariance of the two centred structures. Write its singular value decomposition $H=USV^\top$
([Module 14](14_pca_dimensionality_reduction.ipynb), Section 6.4). Then $\mathrm{tr}(RUSV^\top)=\mathrm{tr}(MS)$ with
$M=V^\top RU$ orthogonal, which is largest when $M=I$, i.e.

$$R=V\,\mathrm{diag}\big(1,\,1,\,d\big)\,U^\top,\qquad d=\operatorname{sign}\det(VU^\top).$$

*In words:* rotate along the singular vectors. If $VU^\top$ has determinant $-1$ it is a reflection, which would overlay a
structure onto its *mirror image*; flipping the sign of the axis with the smallest singular value gives the best proper
rotation instead. The sister course's [rotations notebook](../../../../cryoem/tutorial/BiologyToResearch/notebooks/02_rotations_geometry.ipynb) uses the same algorithm to superpose
subunits in cryo-EM maps.

In [ ]:
# ── 3.1 Rotations, the Kabsch algorithm and RMSD — with a 🧪 check against a known rotation ──
def random_rotation(rng):
    """A uniformly random 3-D rotation (QR decomposition of a Gaussian matrix, signs fixed so that det = +1)."""
    Q, R = np.linalg.qr(rng.standard_normal((3, 3)))
    Q = Q * np.sign(np.diag(R))
    if np.linalg.det(Q) < 0:
        Q[:, 0] = -Q[:, 0]
    return Q


def kabsch(P, Q, allow_reflection=False):
    """Rotation R and translation t minimising Σ‖R p_i + t − q_i‖² (rows of P and Q are matched atoms)."""
    p0, q0 = P.mean(0), Q.mean(0)
    H = (P - p0).T @ (Q - q0)                                          # 3 × 3 cross-covariance
    U, S, Vt = np.linalg.svd(H)
    d = 1.0 if allow_reflection else np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    return R, q0 - p0 @ R.T


def superpose(P, Q, **kw):
    """P moved onto Q by the best rigid motion."""
    R, t = kabsch(P, Q, **kw)
    return P @ R.T + t


def rmsd(P, Q):
    return float(np.sqrt(np.mean(np.sum((P - Q) ** 2, axis=1))))


rng = np.random.default_rng(SEED + 3)
R_true = random_rotation(rng)
P_moved = X_true @ R_true.T + np.array([30.0, -12.0, 5.0]) + rng.normal(0, 0.5, X_true.shape)   # rotated, shifted, noisy
R_hat, _ = kabsch(P_moved, X_true)
angle_err = np.degrees(np.arccos(np.clip((np.trace(R_hat @ R_true) - 1) / 2, -1, 1)))
print(f"RMSD before superposition: {rmsd(P_moved, X_true):.1f} Å | after Kabsch: {rmsd(superpose(P_moved, X_true), X_true):.2f} Å "
      f"(noise 0.5 Å per coordinate → expected ≈ 0.5·√3 = {0.5 * np.sqrt(3):.2f} Å)")
print(f"error of the recovered rotation: {angle_err:.2f}°   (det R = {np.linalg.det(R_hat):+.3f})")
print(f"mirror image: RMSD after a proper rotation {rmsd(superpose(X_mirror, X_true), X_true):.2f} Å | "
      f"if reflections were allowed {rmsd(superpose(X_mirror, X_true, allow_reflection=True), X_true):.2f} Å")

🔍 **What just happened?** Before superposition the RMSD is meaningless (tens of Å, all from the arbitrary placement).
Kabsch recovers the rotation to within about a degree, and the RMSD after superposition is about $0.5\sqrt3$ Å — exactly
the added noise (slightly less, because six fitted parameters absorb a little of it). The mirror image is the instructive
case: it has *identical* internal distances, so if reflections were allowed it would overlay perfectly (RMSD 0); with a
proper rotation, the best we can do is about 10 Å. A mirror-image protein is a different molecule.

**Beyond RMSD.** RMSD averages *squared* deviations, so a few badly placed atoms dominate it: a correct domain with one
floppy tail can look as bad as a wrong fold. Two scores designed to avoid this are used in CASP:

* **TM-score** (Zhang & Skolnick 2004) — after the superposition that *maximises* it,

$$\mathrm{TM}=\max_{R,t}\;\frac1L\sum_{i=1}^{L}\frac{1}{1+(d_i/d_0)^2},\qquad d_0(L)=1.24\,(L-15)^{1/3}-1.8\ \text{Å}.$$

*In words:* each residue scores close to 1 if it is within about $d_0$ (2.9 Å for $L=68$) of its partner and close to 0
if far away, so a residue that is badly misplaced costs at most $1/L$, however far off it is. The length-dependent $d_0$
makes scores comparable across protein sizes: unrelated structures score about 0.17, and TM > 0.5 indicates the same fold
with high probability (Xu & Zhang 2010).

* **lDDT** (local distance difference test; Mariani *et al.* 2013) — no superposition at all:

$$\mathrm{lDDT}=\frac14\sum_{c\in\{0.5,1,2,4\ \text{Å}\}}\frac{\#\{(i,j):\ |d^{\,\text{model}}_{ij}-d^{\,\text{true}}_{ij}|<c,\ d^{\,\text{true}}_{ij}<15\ \text{Å}\}}{\#\{(i,j):\ d^{\,\text{true}}_{ij}<15\ \text{Å}\}}.$$

*In words:* for every pair of residues that are within 15 Å in the true structure, ask whether the model preserves their
distance within 0.5, 1, 2 and 4 Å, and average. Because it compares *internal* distances, lDDT needs no superposition
and judges each residue's local environment; it is computed per residue, too — which is what AlphaFold's pLDDT predicts.

In [ ]:
# ── 3.2 TM-score (simplified search) and lDDT (Cα version) ──
def tm_score(P, Q):
    """TM-score of model P against reference Q (matched residues). Searches superpositions seeded on fragments of length
    L, L/2, L/4 and 8, iteratively re-superposing on the residues within d0 (a simplified version of the TM-score program)."""
    n = len(Q)
    d0 = 1.24 * (n - 15) ** (1 / 3) - 1.8
    best = 0.0
    for w in sorted({n, n // 2, n // 4, 8}):
        for start in range(0, n - w + 1, max(1, w // 2)):
            sel = np.arange(start, start + w)
            for _ in range(20):
                R, t = kabsch(P[sel], Q[sel])                              # superpose on the selected residues ...
                d = np.linalg.norm(P @ R.T + t - Q, axis=1)                # ... and score all residues
                best = max(best, np.mean(1 / (1 + (d / d0) ** 2)))
                new = np.flatnonzero(d < max(d0, np.sort(d)[2]))           # at least 3 residues
                if np.array_equal(new, sel):
                    break
                sel = new
    return float(best)


def lddt(P, Q, per_residue=False, radius=15.0):
    """Cα lDDT of model P against reference Q: fraction of reference distances < radius preserved within 0.5/1/2/4 Å."""
    Dp = np.linalg.norm(P[:, None] - P[None], axis=-1)
    Dq = np.linalg.norm(Q[:, None] - Q[None], axis=-1)
    pairs = (Dq < radius) & ~np.eye(len(Q), dtype=bool)
    kept = np.mean([np.abs(Dp - Dq) < c for c in (0.5, 1.0, 2.0, 4.0)], axis=0) * pairs
    return kept.sum(1) / pairs.sum(1) if per_residue else float(kept.sum() / pairs.sum())


print(f"identical structures: TM = {tm_score(X_true, X_true):.3f}, lDDT = {lddt(X_true, X_true):.3f}")
print(f"mirror image        : TM = {tm_score(X_mirror, X_true):.3f}, lDDT = {lddt(X_mirror, X_true):.3f}  "
      "(lDDT, built from distances only, cannot see chirality!)")

In [ ]:
# ── 3.3 🧪 Which score forgives what? Global noise versus a hinge motion of the last helix ──
def rotate_about(X, idx, pivot, axis, degrees):
    """Rigidly rotate the atoms idx of X about an axis through the point pivot (Rodrigues' formula)."""
    a = axis / np.linalg.norm(axis)
    K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
    ang = np.radians(degrees)
    R = np.eye(3) + np.sin(ang) * K + (1 - np.cos(ang)) * K @ K
    Y = X.copy()
    Y[idx] = (X[idx] - pivot) @ R.T + pivot
    return Y


rng = np.random.default_rng(SEED + 33)
rows3, hinge_models = [], {}
for sigma in (0.5, 1.0, 2.0):
    P = X_true + rng.normal(0, sigma, X_true.shape)
    rows3.append(("global noise", f"σ = {sigma} Å", rmsd(superpose(P, X_true), X_true), tm_score(P, X_true), lddt(P, X_true)))
moving = np.arange(51, L)                                    # the last loop's tail + helix 4 swing about residue 50
for deg in (15, 30, 60, 90):
    P = rotate_about(X_true, moving, X_true[50], np.array([1.0, 0.0, 0.0]), deg)
    hinge_models[deg] = P
    rows3.append(("hinge (helix 4)", f"{deg}°", rmsd(superpose(P, X_true), X_true), tm_score(P, X_true), lddt(P, X_true)))
table3 = pd.DataFrame(rows3, columns=["perturbation", "size", "RMSD (Å)", "TM-score", "lDDT"])
print(table3.round(3).to_string(index=False))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
sub = table3[table3["perturbation"] == "hinge (helix 4)"]
angles = [15, 30, 60, 90]
a1.plot(angles, sub["RMSD (Å)"] / sub["RMSD (Å)"].max(), "o-", color=OKABE_ITO[1], label="RMSD (scaled to its max)")
a1.plot(angles, sub["TM-score"], "s-", color=OKABE_ITO[0], label="TM-score")
a1.plot(angles, sub["lDDT"], "^-", color=OKABE_ITO[2], label="lDDT")
a1.set_xlabel("hinge angle of helix 4 (degrees)"); a1.set_ylim(0, 1.05); a1.legend(fontsize=7)
a1.set_title("RMSD keeps growing; TM-score and lDDT level off")
P60 = superpose(hinge_models[60], X_true)
a2.plot(X_true[:, 0], X_true[:, 2], "-", color="0.6", lw=1.5, label="true")
a2.plot(P60[:, 0], P60[:, 2], "-", color=OKABE_ITO[1], lw=1, label="helix 4 swung by 60°")
a2.set_aspect("equal"); a2.set_xlabel("x (Å)"); a2.set_ylabel("z (Å)"); a2.legend(fontsize=7)
a2.set_title("after the best global superposition")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Read the table in two halves.

* **Global noise** perturbs every residue a little. All three scores degrade together, and lDDT is the strictest: with
  1 Å of noise per coordinate many pairwise distances change by more than 0.5 or 1 Å, so lDDT falls to about 0.7 while the
  TM-score is still about 0.8.
* **A hinge motion** leaves residues 0–50 (51 of 68) perfectly placed and rigidly swings the other 17 — the tail of the
  last loop and helix 4 — about residue 50, like a domain on a flexible linker. RMSD grows steadily with the angle (to
  more than 9 Å at 90°), as if the whole structure were wrong. The TM-score levels off at about 0.76, close to
  $51/68=0.75$ — the fraction of residues still in place — and lDDT levels off near 0.69: the moved segment keeps its
  internal distances and loses only its contacts with the rest.

So RMSD answers "how far apart are the atoms after one global superposition?", while TM-score and lDDT answer "how much
of the structure is right?". For multi-domain proteins and flexible complexes the latter is usually the question.

### 📄 Research connection — Comparing protein structures

- **What the field does.** Optimal superposition by the SVD dates back to Kabsch (1976). Because RMSD is dominated by the worst
    regions and depends on protein length, CASP adopted scores that count well-placed residues: GDT_TS (the average
    fraction of residues within 1, 2, 4 and 8 Å under optimised superpositions; Zemla 2003), the TM-score with its
    length-normalised $d_0$ (Zhang & Skolnick 2004) and, for structural alignment of different sequences, TM-align
    (Zhang & Skolnick 2005). The superposition-free lDDT (Mariani *et al.* 2013) became a standard of CASP and CAMEO and
    the target of AlphaFold's confidence head. For complexes, interface scores such as DockQ (Basu & Wallner 2016) are
    used.
- **What this notebook builds.** Kabsch superposition with reflection handling, RMSD, a simplified TM-score search over fragment-seeded
    superpositions, and Cα-lDDT (global and per residue), compared on global noise and on a hinge motion with known
    truth.
- **Where the frontier is.** Comparing *ensembles* rather than single structures (proteins move), scoring complexes and interfaces,
    all-atom and ligand-pose accuracy, and searching structure databases of hundreds of millions of entries fast enough
    (Foldseek encodes local structure as a 20-letter "3Di" alphabet so that sequence-search methods can be used; van
    Kempen *et al.* 2024).
- **Read next.** Kabsch 1976 (Acta Crystallogr A); Zhang & Skolnick 2004 (Proteins); Mariani *et al.* 2013 (Bioinformatics).

✏️ **Your turn.** Displace only the third loop (residues 50–53) outwards by 6 Å, without moving anything else. Which of
the three scores barely changes, and which one is most sensitive? Why might that matter when evaluating predicted loops?

---
## 4 · From distances to coordinates

> 📍 **Where we are.** Prediction methods from NMR to AlphaFold 1 produce *distances* (or contacts) between residues, not
> coordinates. Distances are invariant to rotation and translation — which is convenient for prediction — but then we
> need a way back to 3-D. That is **distance geometry**.

🧮 **Math refresher — classical multidimensional scaling (MDS; Torgerson 1952).** If $X$ ($L\times3$) holds centred
coordinates, the Gram matrix $B=XX^\top$ contains all dot products $x_i\cdot x_j$, and distances and dot products are
linked by $d_{ij}^2=\lVert x_i\rVert^2+\lVert x_j\rVert^2-2\,x_i\cdot x_j$. "Double centring" inverts this:

$$B=-\tfrac12\,J\,D^{(2)}J,\qquad J=I-\tfrac1L\mathbf 1\mathbf 1^\top,$$

where $D^{(2)}$ holds the squared distances. Then the eigen-decomposition $B=V\Lambda V^\top$ gives the coordinates
$X=V_3\Lambda_3^{1/2}$ from the three largest eigenvalues and their eigenvectors.

*In words:* turn distances into dot products, then read coordinates off the top three eigenvectors. For a genuine 3-D
structure $B$ has rank 3: exactly three non-zero eigenvalues.

> 🔗 **Builds on [Module 14 · PCA](14_pca_dimensionality_reduction.ipynb), Sections 3 and 6.** PCA takes the
> eigenvectors of $X^\top X$ (features × features); classical MDS takes those of $XX^\top$ (points × points). For
> Euclidean distances the two give the same coordinates — MDS is PCA performed when you only know the distances.

**The mirror ambiguity.** Distances do not change under reflection, so MDS can only return the structure *or its mirror
image* — which one depends on arbitrary signs of eigenvectors. The distance matrix alone cannot tell them apart (Section
3: TM-score 0.3, but lDDT 1.0). Physics can: real α-helices are right-handed, so we check the sign of the helix
pseudo-torsions and reflect the solution if they come out negative.

In [ ]:
# ── 4.1 Classical MDS on the exact distance matrix, and fixing the mirror image ──
def classical_mds(D, dim=3):
    """Coordinates whose distances best match D: eigen-decomposition of the double-centred squared distances."""
    n = len(D)
    J = np.eye(n) - 1 / n
    B = -0.5 * J @ (D ** 2) @ J
    w, V = np.linalg.eigh(B)
    top = np.argsort(w)[::-1][:dim]
    return V[:, top] * np.sqrt(np.maximum(w[top], 0)), np.sort(w)[::-1]


def fix_handedness(X):
    """Reflect X (z → −z) if most helix pseudo-torsions are negative, i.e. if its helices are left-handed."""
    tau = np.concatenate([pseudo_torsions(X)[h[0]:h[-1] - 2] for h in HELICES])
    return (X, False) if np.mean(tau > 0) >= 0.5 else (X * np.array([1, 1, -1]), True)


X_mds, eigvals = classical_mds(D_true)
print("largest eigenvalues of B:", np.round(eigvals[:6], 1), "→ exactly three are non-zero: the data are 3-D")
print(f"MDS solution as returned: RMSD to truth {rmsd(superpose(X_mds, X_true), X_true):.2f} Å; "
      f"helix torsions {np.round(helix_torsions(X_mds), 0)}")
X_fixed, flipped = fix_handedness(X_mds)
print(f"after the handedness check (reflected: {flipped}): RMSD {rmsd(superpose(X_fixed, X_true), X_true):.2f} Å, "
      f"TM-score {tm_score(X_fixed, X_true):.3f}")

🔍 **What just happened?** The double-centred matrix has three non-zero eigenvalues and the rest are zero to rounding: the
68 × 68 distance matrix encodes exactly three dimensions. The MDS solution came out as the **mirror image** (helix
torsions $-50°$; RMSD about 10 Å even after the best rotation). One reflection, triggered by the handedness check, gives
back the true structure exactly (RMSD 0.00 Å, TM-score 1).

Real distance information is noisy and incomplete. NMR restraints only exist for short distances; predicted distances
have errors of several Å. Let us test both problems.

In [ ]:
# ── 4.2 🧪 Noisy distances, and incomplete distances completed by shortest paths ──
rng = np.random.default_rng(SEED + 4)
print("noisy but complete distance matrix:")
for sigma in (1.0, 2.0, 4.0):
    N = rng.normal(0, sigma, (L, L)); N = (N + N.T) / np.sqrt(2); np.fill_diagonal(N, 0)
    Xn, _ = fix_handedness(classical_mds(np.abs(D_true + N))[0])
    print(f"   noise σ = {sigma} Å on every distance → RMSD {rmsd(superpose(Xn, X_true), X_true):.2f} Å, TM-score {tm_score(Xn, X_true):.2f}")
KNOWN = (D_true < 12.0) & ~np.eye(L, dtype=bool)                       # like NMR: only short distances are measured
N = rng.normal(0, 1.0, (L, L)); N = (N + N.T) / np.sqrt(2)
D_obs = np.where(KNOWN, np.abs(D_true + N), 0.0)                      # 0 = unknown
D_completed = shortest_path(D_obs, directed=False)                     # unknown distance ≈ shortest path through known ones
X_sp, _ = fix_handedness(classical_mds(D_completed)[0])
print(f"\nonly distances < 12 Å known ({100 * KNOWN.mean():.0f} % of pairs), noise σ = 1 Å; missing ones from shortest paths:")
fill_err = (D_completed - D_true)[~KNOWN & (SEP > 0)]
print(f"   MDS → RMSD {rmsd(superpose(X_sp, X_true), X_true):.2f} Å, TM-score {tm_score(X_sp, X_true):.2f}")
print(f"   filled-in distances: mean error {fill_err.mean():+.1f} Å, mean absolute error {np.abs(fill_err).mean():.1f} Å")

🔍 **What just happened?** MDS is remarkably tolerant of noise: every coordinate is estimated from all 67 distances of its
residue, so independent errors average out (RMSD 0.56 Å for 1 Å of noise, still TM ≈ 0.66 for 4 Å). Missing distances
are a harder problem. Only 32 % of the pairs are "measured" here. Filling the rest with **shortest paths** through the
known distances is the idea behind "bound smoothing" in classical distance geometry (and behind Isomap): for exact
distances the triangle inequality guarantees that a path is never shorter than the direct distance, so the fill-in can
only overestimate. With noisy measurements the shortest of many paths also collects negative errors, and here the
filled-in distances are off by 1.2 Å on average (and slightly too short overall, by 0.5 Å). MDS on this matrix gives the
correct fold (TM ≈ 0.8) but not the details (RMSD 1.6 Å).

The remedy is to **optimise** coordinates directly so that they match only the distances we trust — a non-convex problem
that we solve by gradient descent ([Module 04](04_calculus_and_optimization.ipynb)), with PyTorch computing the gradients:

$$E(X)=\sum_{(i,j)\ \text{known}}\big(\lVert x_i-x_j\rVert-d^{\,\text{obs}}_{ij}\big)^2+10\sum_i\big(\lVert x_i-x_{i+1}\rVert-3.8\big)^2 .$$

*In words:* the "stress" penalises every known distance the model gets wrong, and a stiff extra term holds consecutive
Cα atoms 3.8 Å apart. We start either from random coordinates or from the shortest-path MDS solution.

In [ ]:
# ── 4.3 🧪 Gradient-based embedding: random starts versus an MDS start ──
BOND = torch.tensor(SEP == 1)


def stress_embedding(X0, D_obs, known, n_steps=2000, lr=0.05):
    """Minimise the stress on known distances + 3.8 Å bond restraints by Adam, for a batch of starting structures X0."""
    x = torch.tensor(X0, dtype=torch.float32).requires_grad_(True)
    Dt, K = torch.tensor(D_obs, dtype=torch.float32), torch.tensor(known)
    opt = torch.optim.Adam([x], lr=lr)
    for _ in range(n_steps):
        D = torch.sqrt(((x[..., :, None, :] - x[..., None, :, :]) ** 2).sum(-1) + 1e-8)
        E = ((D - Dt) ** 2 * K).sum((-1, -2)) / 2 + 10 * ((D - 3.8) ** 2 * BOND).sum((-1, -2)) / 2
        opt.zero_grad(); E.sum().backward(); opt.step()
    return x.detach().numpy(), E.detach().numpy()


torch.manual_seed(SEED)
t0 = time.time()
starts = np.concatenate([np.random.default_rng(SEED + 43).normal(0, 10, (8, L, 3)), X_sp[None]])   # 8 random + MDS
X_emb, E_emb = stress_embedding(starts, D_obs, KNOWN)
rows4 = []
for k, (X, E) in enumerate(zip(X_emb, E_emb)):
    Xf, flipped = fix_handedness(X)
    tau = helix_torsions(X)
    rows4.append({"start": "MDS (shortest paths)" if k == 8 else f"random {k + 1}", "final stress": float(E),
                  "right-handed helices (of 4)": int(np.sum(tau * (-1 if flipped else 1) > 0)),
                  "RMSD (Å)": rmsd(superpose(Xf, X_true), X_true), "TM-score": tm_score(Xf, X_true)})
print(f"9 embeddings optimised together in {time.time() - t0:.1f} s")
print(pd.DataFrame(rows4).round(2).to_string(index=False))

🔍 **What just happened?** The same objective, optimised from nine starting points, gives very different answers. Every
random start ends in a **local minimum** with high stress (above 2,000), a wrong fold (TM-score 0.20–0.44) and, in most
runs, helices of mixed handedness — the chain got tangled early and gradient descent cannot untangle it. Started from the
MDS solution, the same optimiser reaches a stress of about 600 and an almost perfect structure (TM-score 0.94, RMSD
0.76 Å). The lowest stress also identified the best structure — a criterion that needs no knowledge of the truth.

> 🔗 **Builds on [Module 04](04_calculus_and_optimization.ipynb), Section 5 ("Mount Fuji or the Himalayas?").** Folding
> energies are the Himalayas: many valleys. Good initialisation, many restarts and annealing (the simulated annealing of
> [Module 21](21_bayesian_learning_mcmc.ipynb)'s MCMC, run with a decreasing temperature) are the standard defences — and
> part of AlphaFold 2's design is "recycling" its own prediction as the starting point for another pass.

⚠️ **Pitfall.** Mixed handedness is a symptom of a bad local minimum: the distance restraints alone cannot fix chirality,
so each helix "chose" a hand independently. Reflecting the whole structure cannot repair that; only a better optimum (or
chirality restraints) can.

✏️ **Your turn.** Repeat 4.3 with only distances below 8 Å known (as for a contact map). How do the MDS start and the
best random start compare now?

---
## 5 · From predicted contacts to 3-D

> 📍 **Where we are.** In Module 37 direct-coupling analysis (DCA) predicted contacts from a family alignment. Here we
> close the loop: family → predicted contacts → 3-D structure → comparison with the truth. This is the EVfold pipeline
> of Marks *et al.* (2011), in miniature.

**The family.** `simulate_protein_family` draws 1,000 aligned sequences from a Potts model whose couplings sit on the
contacts of our protein — so co-varying columns really are contacts. We rank residue pairs with **mean-field
DCA** exactly as in [Module 37](37_protein_sequences_language_models.ipynb), Section 4: invert the covariance matrix of
the one-hot alignment columns, measure each pair's coupling strength by its Frobenius norm, and apply the average product
correction (APC).

In [ ]:
# ── 5.1 A SIMULATED family and mean-field DCA contact predictions (Module 37) ──
def mean_field_dca_scores(msa, q=20, pseudocount=0.5):
    """APC-corrected Frobenius norms of the mean-field couplings J = −C⁻¹ (Morcos et al. 2011), L × L."""
    n, length = msa.shape
    X1 = np.eye(q)[msa].reshape(n, -1)
    fij = (X1.T @ X1 / n).reshape(length, q, length, q).transpose(0, 2, 1, 3)
    fi = (1 - pseudocount) * X1.mean(0).reshape(length, q) + pseudocount / q
    fij = (1 - pseudocount) * fij + pseudocount / q ** 2
    for i in range(length):
        fij[i, i] = np.diag(fi[i])
    C = (fij - fi[:, None, :, None] * fi[None, :, None, :])[:, :, :q - 1, :q - 1]
    J = -np.linalg.inv(C.transpose(0, 2, 1, 3).reshape(length * (q - 1), -1)).reshape(length, q - 1, length, q - 1)
    J = J.transpose(0, 2, 1, 3)
    J = J - J.mean(2, keepdims=True) - J.mean(3, keepdims=True) + J.mean((2, 3), keepdims=True)   # zero-sum gauge
    S = np.sqrt((J ** 2).sum((2, 3))); np.fill_diagonal(S, 0)
    row = S.sum(1) / (length - 1)
    S = S - np.outer(row, row) / (S.sum() / (length * (length - 1)))                              # APC
    np.fill_diagonal(S, 0)
    return S


IU6 = np.triu_indices(L, 6)


def top_contacts(S, k):
    """Symmetric boolean matrix of the k highest-scoring pairs with |i − j| ≥ 6."""
    M = np.zeros((L, L), bool)
    best = np.argsort(-S[IU6])[:k]
    M[IU6[0][best], IU6[1][best]] = True
    return M | M.T


rng = np.random.default_rng(SEED + 5)
t0 = time.time()
family = simulate_protein_family(rng, contacts, n_seqs=1000)
msa = family["msa"]
scores = {"1,000 sequences": mean_field_dca_scores(msa), "300 sequences": mean_field_dca_scores(msa[:300])}
print(f"SIMULATED family: {msa.shape[0]} sequences × {msa.shape[1]} positions; family + DCA in {time.time() - t0:.1f} s")
print(f"true long-range contacts: {LONG.sum() // 2} — so precision of the top L = 68 pairs can be at most {LONG.sum() / 2 / L:.2f}")
for name, S in scores.items():
    prec = {f"top {k}": np.mean(LONG[IU6][np.argsort(-S[IU6])[:k]]) for k in (L // 4, L // 2, L)}
    print(f"   {name:15s} precision: " + "  ".join(f"{k} = {v:.2f}" for k, v in prec.items()))

🔍 **What just happened?** With 1,000 sequences (about 15 per residue) DCA's strongest pairs are all genuine contacts;
since the protein has only 38 long-range contacts, the top $L=68$ list necessarily contains at least 30 false ones. With
300 sequences (about 4 per residue) even the top $L/2$ list is only partly right, as Module 37's depth curve predicted.

**Restraint-based folding.** We now turn a contact list into a structure. As in real contact-guided folding (EVfold used
predicted secondary structure as extra restraints), we assume the **secondary structure is known** — secondary-structure
predictors are accurate for helices — and restrain residues $i$ and $i+1..i+4$ inside each helix to ideal α-helix
distances (3.8, 5.4, 5.0 and 6.2 Å). The energy is

$$E(X)=10\!\!\sum_{\text{local}}\!\!(d_{ij}-d^{\,\text{ideal}}_{ij})^2+\!\!\sum_{\text{contacts}}\!\!\max(0,d_{ij}-8)^2
+\sum_{|i-j|\ge2}\max(0,4-d_{ij})^2+0.05\!\!\sum_{\text{other pairs}}\!\!\max(0,8-d_{ij})^2 .$$

*In words:* keep chain and helices ideal; pull every predicted contact below 8 Å; forbid two Cα atoms from coming closer
than 4 Å (steric clash); and gently push all pairs *not* predicted to be in contact apart. Following Section 4, we start
from MDS on shortest paths (each predicted contact counted as a 6 Å link), refine by gradient descent from several
slightly perturbed starts, keep the lowest-energy model, and fix the mirror image with the helix handedness.

In [ ]:
# ── 5.2 Restraint-based folding (batched in PyTorch) ──
IDEAL_HELIX = {k: float(np.hypot(2 * 2.3 * np.sin(np.radians(50 * k)), 1.5 * k)) for k in range(1, 5)}   # ideal α-helix
LOCAL = np.full((L, L), np.nan)
for h in HELICES:
    for k in range(1, 5):
        for i in h[:-k]:
            LOCAL[i, i + k] = LOCAL[i + k, i] = IDEAL_HELIX[k]
for i in range(L - 1):
    if np.isnan(LOCAL[i, i + 1]):
        LOCAL[i, i + 1] = LOCAL[i + 1, i] = 3.8                      # loop residues: only the 3.8 Å chain constraint
LOCAL_T = torch.tensor(np.nan_to_num(LOCAL), dtype=torch.float32)
LOCAL_M, SEP_T = LOCAL_T > 0, torch.tensor(SEP)


def mds_start(contact_mask, contact_length=6.0):
    """Initial coordinates: MDS of shortest-path distances through local restraints and predicted contacts."""
    G = np.nan_to_num(LOCAL)
    G[contact_mask & np.isnan(LOCAL)] = contact_length
    X0, _ = classical_mds(shortest_path(G, directed=False))
    return X0


def fold(starts, contact_masks, n_steps=1500, lr=0.05, w_repel=0.05):
    """Minimise the restraint energy for a batch of starts (B × L × 3) with their contact masks (B × L × L)."""
    x = torch.tensor(starts, dtype=torch.float32).requires_grad_(True)
    C = torch.tensor(contact_masks) & ~LOCAL_M
    other = ~torch.tensor(contact_masks) & (SEP_T >= 3) & ~LOCAL_M
    opt, eye = torch.optim.Adam([x], lr=lr), torch.eye(L) * 100
    for _ in range(n_steps):
        D = torch.cdist(x, x) + eye
        E = (10 * ((D - LOCAL_T) ** 2 * LOCAL_M).sum((-1, -2)) + (torch.relu(D - 8.0) ** 2 * C).sum((-1, -2))
             + (torch.relu(4.0 - D) ** 2 * (SEP_T >= 2)).sum((-1, -2))
             + w_repel * (torch.relu(8.0 - D) ** 2 * other).sum((-1, -2))) / 2
        opt.zero_grad(); E.sum().backward(); opt.step()
    return x.detach().numpy(), E.detach().numpy()


RESTRAINT_SETS = {
    "secondary structure only": np.zeros((L, L), bool),
    "+ all true long-range contacts": LONG,
    "+ DCA top L/2 (1,000 seqs)": top_contacts(scores["1,000 sequences"], L // 2),
    "+ DCA top L (1,000 seqs)": top_contacts(scores["1,000 sequences"], L),
    "+ DCA top L/2 (300 seqs)": top_contacts(scores["300 sequences"], L // 2),
}
print({k: int(v.sum() // 2) for k, v in RESTRAINT_SETS.items()}, "contacts per set")

In [ ]:
# ── 5.3 🧪 Fold with each restraint set and compare with the truth ──
N_STARTS = 6
rng = np.random.default_rng(SEED + 53)
starts, masks, names = [], [], []
for name, M in RESTRAINT_SETS.items():
    base = mds_start(M)
    for _ in range(N_STARTS):                                          # same restraints, slightly different starts
        starts.append(base + rng.normal(0, 1.0, base.shape)); masks.append(M); names.append(name)
t0 = time.time()
models, energies = fold(np.stack(starts), np.stack(masks))
print(f"{len(models)} folding runs ({N_STARTS} per restraint set) optimised together in {time.time() - t0:.1f} s")
names, rows5, best_models = np.array(names), [], {}
for name, M in RESTRAINT_SETS.items():
    idx = np.flatnonzero(names == name)
    b = idx[np.argmin(energies[idx])]                                  # the prediction = the lowest-energy model
    Xb, flipped = fix_handedness(models[b])
    best_models[name] = Xb
    prec = np.mean(LONG[M]) if M.any() else np.nan
    tm_all = [tm_score(fix_handedness(models[i])[0], X_true) for i in idx]
    rows5.append({"restraints": name, "contact precision": prec, "energy": float(energies[b]), "reflected": flipped,
                  "TM-score": tm_score(Xb, X_true), "TM range (all starts)": f"{min(tm_all):.2f}–{max(tm_all):.2f}",
                  "TM of mirror": tm_score(Xb * np.array([1, 1, -1]), X_true),
                  "RMSD (Å)": rmsd(superpose(Xb, X_true), X_true), "lDDT": lddt(Xb, X_true)})
table5 = pd.DataFrame(rows5).set_index("restraints")
print(table5.round(2).to_string())

In [ ]:
# ── 5.4 Picture: predicted models superposed on the truth ──
fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, name in zip(axes, ["+ all true long-range contacts", "+ DCA top L/2 (1,000 seqs)", "+ DCA top L (1,000 seqs)",
                           "+ DCA top L/2 (300 seqs)"]):
    P = superpose(best_models[name], X_true)
    ax.plot(X_true[:, 0], X_true[:, 2], color="0.7", lw=2.5, label="true")
    ax.plot(P[:, 0], P[:, 2], color=OKABE_ITO[0], lw=1, label="model")
    ax.set_title(f"{name}\nTM = {table5.loc[name, 'TM-score']:.2f}", fontsize=8)
    ax.set_xlim(-16, 16); ax.set_ylim(-17, 17); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=7, loc="lower center")
fig.suptitle("Models from SIMULATED coevolution contacts (side view after superposition)", fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.92)); plt.show()

🔍 **What just happened?** Read the table from top to bottom.

* **Secondary structure alone** produces four good helices but no fold: without long-range attraction the gentle
  repulsion pushes them apart (RMSD almost 40 Å, TM-score 0.25). Long-range information is what turns helices into a fold.
* **All 38 true long-range contacts** (the oracle) give a model with TM-score 0.91, RMSD 1.0 Å and lDDT 0.87, with all
  restraints satisfied (energy ≈ 0).
* **DCA's top $L/2$ contacts from 1,000 sequences** are all correct, yet the model is less accurate (TM-score 0.79,
  RMSD 2.5 Å): 34 contacts leave more freedom than 38, and this run ended in a slightly strained minimum (energy 31).
  The fold is right; the details are not. (Even the oracle's six runs span TM 0.86–0.91: a set of "< 8 Å" restraints
  does not pin down a structure exactly.)
* **More is not better.** DCA's top $L$ list from the same alignment contains 30 false contacts. Forcing them below 8 Å
  distorts the structure: TM-score 0.36, the highest energy of all.
* **A shallow alignment** (300 sequences, contact precision 0.65) also gives TM-score 0.36: the fold is lost. Notice
  that here the lowest-energy run was not the most accurate one (the range reaches 0.44) — with wrong restraints, the
  energy no longer measures accuracy.
* The **mirror** column shows how much the handedness check matters: reflected models score only 0.23–0.31, and three of
  the five predictions came out of the optimiser as mirror images. Only the helix pseudo-torsions tell us which hand is
  real.

These are the two classic lessons of contact-guided folding: **precision matters more than quantity**, and **alignment
depth limits everything**. They are why the field moved, from about 2017, to deep networks that predict contacts and
distances far more precisely from the same alignments (RaptorX-Contact, AlphaFold 1, trRosetta) — and then to AlphaFold 2,
which predicts the structure end to end.

✏️ **Your turn.** Make the contact restraint *robust*: replace $\max(0,d-8)^2$ by a penalty that saturates, e.g.
$1-\exp(-\max(0,d-8)^2/25)$, so that a violated (possibly false) contact costs at most 1 (multiply by a weight of about 20).
Does the top-$L$ model improve?

### 📄 Research connection — From coevolution to deep learning

- **What the field does.** Coevolution-based contacts became accurate enough to fold proteins in 2011: EVfold (Marks *et al.* 2011)
    combined DCA contacts with predicted secondary structure as distance restraints and built correct folds for
    proteins of unknown structure. Deep networks then learned to predict contacts and full distance distributions
    ("distograms") from alignments: AlphaFold 1 (Senior *et al.* 2020) folded proteins by gradient descent on a potential
    built from predicted distograms — the same recipe as this section with far better restraints — and trRosetta (Yang
    *et al.* 2020) predicted distances and inter-residue orientations and folded with Rosetta.
- **What this notebook builds.** The whole classical pipeline on a simulated family with known structure: mean-field DCA contacts,
    secondary-structure restraints, MDS initialisation on shortest paths, batched gradient-based folding, mirror
    resolution by helix handedness, and evaluation with TM-score, RMSD and lDDT against the truth.
- **Where the frontier is.** End-to-end predictors (AlphaFold 2, RoseTTAFold) made restraint-based folding largely historical for
    single chains, but the questions remain: families with few sequences, designed and orphan proteins (ESMFold works
    from a single sequence via a language model; Lin *et al.* 2023), and complexes, where coevolution *between* proteins
    requires pairing sequences across families. AlphaFold DB (Varadi *et al.* 2022) and the ESM Metagenomic Atlas made
    predicted structures available for hundreds of millions of proteins.
- **Read next.** Marks *et al.* 2011 (PLoS ONE); Senior *et al.* 2020 (Nature); Yang *et al.* 2020 (PNAS).

---
## 6 · Confidence: knowing when a model is wrong

> 📍 **Where we are.** In real use there is no true structure to compare with. A predictor must say *how much to trust
> each part of its model*. AlphaFold's pLDDT is a per-residue prediction of lDDT (0–100); its PAE predicts, for every
> pair of residues, the position error of one when the model is aligned on the other.

A simple, model-agnostic way to estimate confidence is an **ensemble**: fold the protein many times with slightly
different inputs — here, each run uses a random 80 % of the predicted contacts (a bootstrap of the restraints), a random
contact length for the start and a jittered start — and measure how much the models agree. Where the restraints
determine the structure, all runs agree; where they do not, the runs scatter. We define a **pLDDT-like score**: the
average per-residue lDDT of each ensemble member scored against every other member — no true structure needed.

In [ ]:
# ── 6.1 Bootstrap ensembles and a self-consistency confidence score ──
N_MEMBERS = 10
CONDITIONS = ["+ DCA top L/2 (1,000 seqs)", "+ DCA top L (1,000 seqs)", "+ DCA top L/2 (300 seqs)"]
rng = np.random.default_rng(SEED + 6)
starts, masks = [], []
for name in CONDITIONS:
    pairs = np.argwhere(np.triu(RESTRAINT_SETS[name]))
    for _ in range(N_MEMBERS):
        keep = pairs[rng.random(len(pairs)) < 0.8]                     # bootstrap-like subsample of the contacts
        M = np.zeros((L, L), bool); M[keep[:, 0], keep[:, 1]] = True; M |= M.T
        starts.append(mds_start(M, contact_length=rng.uniform(5, 7)) + rng.normal(0, 1.0, (L, 3))); masks.append(M)
t0 = time.time()
ens_all, _ = fold(np.stack(starts), np.stack(masks))
print(f"{len(ens_all)} ensemble members folded in {time.time() - t0:.1f} s")
ensembles, rows6 = {}, []
for c, name in enumerate(CONDITIONS):
    E = np.stack([fix_handedness(x)[0] for x in ens_all[c * N_MEMBERS:(c + 1) * N_MEMBERS]])
    pair = np.array([[lddt(E[a], E[b], per_residue=True) for b in range(N_MEMBERS)] for a in range(N_MEMBERS)])
    off_diag = ~np.eye(N_MEMBERS, dtype=bool)
    confidence = pair[off_diag].mean(0)                                # per residue: agreement with the other members
    medoid = E[np.argmax(pair.mean(2)[:, :] @ np.ones(N_MEMBERS))]       # the member that agrees best with the rest
    true_lddt = lddt(medoid, X_true, per_residue=True)                 # what the confidence tries to predict
    R, t = kabsch(medoid, X_true)
    error = np.linalg.norm(medoid @ R.T + t - X_true, axis=1)          # per-residue error after superposition
    ensembles[name] = dict(members=E, medoid=medoid, confidence=confidence, true_lddt=true_lddt, error=error)
    rows6.append({"restraints": name, "mean confidence (pLDDT-like)": confidence.mean(), "true lDDT of medoid": true_lddt.mean(),
                  "TM of medoid": tm_score(medoid, X_true), "ρ(confidence, true lDDT)": spearmanr(confidence, true_lddt)[0],
                  "ρ(confidence, −error)": spearmanr(confidence, -error)[0]})
print(pd.DataFrame(rows6).set_index("restraints").round(2).to_string())

In [ ]:
# ── 6.2 Picture: does the confidence predict the error? ──
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [2, 1]})
show = "+ DCA top L (1,000 seqs)"
for h in HELICES:
    a1.axvspan(h[0], h[-1], color=OKABE_ITO[5], alpha=0.12, lw=0)
a1.plot(ensembles[show]["confidence"], color=OKABE_ITO[0], lw=1.5, label="confidence (ensemble self-consistency)")
a1.plot(ensembles[show]["true_lddt"], color=OKABE_ITO[1], lw=1.5, ls="--", label="true per-residue lDDT")
a1.set_xlabel("residue (shaded = helix)"); a1.set_ylabel("lDDT scale (0–1)"); a1.set_ylim(0, 1); a1.legend(fontsize=7)
a1.set_title(f"per residue — {show}")
for name, mk, col in zip(CONDITIONS, "os^", OKABE_ITO[:3]):
    e = ensembles[name]
    a2.scatter(e["confidence"].mean(), e["true_lddt"].mean(), s=60, marker=mk, color=col, label=name)
a2.plot([0, 1], [0, 1], "k:", lw=1)
a2.set_xlim(0.3, 1); a2.set_ylim(0.3, 1); a2.set_xlabel("mean confidence"); a2.set_ylabel("mean true lDDT")
a2.set_title("per model: calibration"); a2.legend(fontsize=6, loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Two levels of evaluation:

* **Per model** (right panel), the ensemble's average self-consistency tracks the true average lDDT closely in all three
  conditions (0.79 vs 0.78, 0.46 vs 0.49, 0.51 vs 0.53): high when the restraints were good, low when they were not. The
  ensemble "knew" which predictions to distrust — without seeing the truth. This is the property that makes pLDDT
  useful in practice: low-confidence models are usually wrong.
* **Per residue** (left panel and the ρ columns), the agreement is weaker: confidence and true lDDT are moderately
  correlated for the two poor models (Spearman ρ ≈ 0.5–0.6) and hardly at all for the good model (ρ ≈ 0.2), where errors
  are small everywhere and the differences between residues are mostly noise.

⚠️ **Pitfall — consistent is not correct.** An ensemble can agree on a wrong answer if all members share the same
mistake — for example, false contacts kept in most bootstrap samples, or (for AlphaFold) a misleading alignment or a
memorised template. Confidence measures *precision*, not *accuracy*; it was calibrated on structures like those seen in
training, and it can be over-confident far from them.

✏️ **Your turn.** Lower the bootstrap fraction from 0.8 to 0.5. Do the ensembles spread more, and does the per-residue
correlation improve or get worse?

### 📄 Research connection — Confidence of structure predictions

- **What the field does.** AlphaFold 2 outputs pLDDT, a per-residue prediction of the lDDT-Cα of its own model, and PAE (predicted
    aligned error) for every residue pair (Jumper *et al.* 2021); a related pTM score summarises global fold confidence.
    For the human proteome, low-pLDDT regions overlapped strongly with regions predicted to be intrinsically disordered
    (Tunyasuvunakool *et al.* 2021), and PAE is widely used to judge the relative placement of domains and the
    plausibility of predicted complexes.
- **What this notebook builds.** An ensemble-based, model-agnostic confidence: bootstrap the restraints, refold, and score each residue by
    the self-consistency lDDT of the ensemble; checked against the true per-residue lDDT, per residue and per model.
- **Where the frontier is.** Calibration of confidence for protein complexes, for designed proteins (where high confidence is used as
    a filter, Module 39) and for mutants; distinguishing "disordered" from "uncertain"; and confidence for multiple
    conformations — current predictors give one structure with one confidence, while many proteins switch between
    states.
- **Read next.** Jumper *et al.* 2021 (Nature); Tunyasuvunakool *et al.* 2021 (Nature).

---
## 7 · How AlphaFold works (conceptually)

> 📍 **Where we are.** You have built every classical ingredient: coevolution (Module 37), contacts and distances, distance
> geometry, optimisation, structure comparison and confidence. AlphaFold 2 (Jumper *et al.* 2021) replaces the hand-made
> pipeline with one network trained end to end on the PDB. This section explains its parts in terms of what you know.

In [ ]:
# ── 7.1 A schematic of AlphaFold 2 (drawn with Matplotlib) ──
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.set_xlim(0, 12); ax.set_ylim(-0.4, 4); ax.axis("off")


def box(x, y, w, h, text, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", fc=color, ec="0.3", lw=1))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=8)


def arrow(x0, y0, x1, y1, **kw):
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=12, color="0.25", lw=1.2, **kw))


box(0.1, 1.6, 1.4, 0.8, "input\nsequence", "#eeeeee")
box(1.9, 2.5, 1.9, 0.9, "MSA search\n(genetic databases)\n+ templates", "#d6e9f8")
box(1.9, 0.6, 1.9, 0.9, "pair features\n(residue i, j)", "#d6e9f8")
box(4.3, 2.5, 1.9, 0.9, "MSA representation\nN_seq × L × c_m", "#cfe8d5")
box(4.3, 0.6, 1.9, 0.9, "pair representation\nL × L × c_z", "#cfe8d5")
box(6.7, 1.0, 1.9, 2.0, "Evoformer\n(48 blocks)\nrow/column attention,\nouter product mean,\ntriangle updates", "#fde3c8")
box(9.1, 1.0, 1.6, 2.0, "structure module\n(8 blocks)\ninvariant point\nattention on\nresidue frames", "#f6d1e3")
box(10.95, 1.5, 1.0, 1.0, "3-D model\n+ pLDDT\n+ PAE", "#eeeeee")
for (x0, y0, x1, y1) in [(1.5, 2.0, 1.9, 2.9), (1.5, 2.0, 1.9, 1.05), (3.8, 2.95, 4.3, 2.95), (3.8, 1.05, 4.3, 1.05),
                         (6.2, 2.95, 6.7, 2.5), (6.2, 1.05, 6.7, 1.5), (8.6, 2.0, 9.1, 2.0), (10.7, 2.0, 10.95, 2.0)]:
    arrow(x0, y0, x1, y1)
arrow(5.25, 2.5, 5.25, 1.5, connectionstyle="arc3,rad=0")                     # MSA → pair (outer product mean)
ax.text(5.35, 2.0, "outer product mean", fontsize=7, va="center")
arrow(11.45, 1.5, 5.25, 0.55, connectionstyle="arc3,rad=-0.2", linestyle="--")
ax.text(8.4, -0.3, "recycling (3×): outputs fed back as inputs", fontsize=8, ha="center", style="italic")
ax.set_title("AlphaFold 2 in one picture (simplified)", fontsize=10)
plt.tight_layout(); plt.show()

**1 · Inputs.** The query sequence is searched against large sequence databases to build a multiple sequence alignment
(MSA) — the coevolution signal of Module 37 — and optionally against the PDB for structural templates.

**2 · Two representations.** AlphaFold keeps an **MSA representation** (one vector per sequence and position) and a
**pair representation** (one vector per residue pair $(i,j)$) — think of the pair representation as a learned, rich
version of the contact and distance maps of Sections 2–5.

**3 · The Evoformer** (48 blocks) updates both:
* *row-wise attention* within each sequence of the MSA, biased by the pair representation, and *column-wise attention*
  within each alignment column across sequences ([Module 24](24_attention_transformers.ipynb)'s attention, on a 2-D grid);
* an *outer product mean* sends information from the MSA to the pair representation — learned coevolution statistics,
  the role DCA's couplings played in Section 5;
* **triangle updates**: the representation of pair $(i,j)$ is updated using the pairs $(i,k)$ and $(k,j)$ for all $k$,
  by multiplicative updates and by attention over $k$. The motivation is geometric consistency. Distances obey the
  **triangle inequality**, $d_{ij}\le d_{ik}+d_{kj}$: if $i$ is close to $k$ and $k$ is close to $j$, then $i$ cannot be
  far from $j$. A single list of predicted pairs does not know that (Section 4 had to *repair* missing distances with
  shortest paths); triangle updates let the network reason about triples of residues.

In [ ]:
# ── 7.2 Triangle inequality: real distance maps obey it, noisy predicted ones do not ──
def triangle_violations(D, tol=0.0):
    """Fraction of triples (i, k, j) with d_ij > d_ik + d_kj + tol."""
    viol = D[:, None, :] > D[:, :, None] + D[None, :, :] + tol          # [i, k, j] compares d_ij with d_ik + d_kj
    return float(viol.mean())


rng = np.random.default_rng(SEED + 72)
print(f"true distance map            : {100 * triangle_violations(D_true):.3f} % of triples violate d_ij ≤ d_ik + d_kj")
for sigma in (2.0, 4.0):
    N = rng.normal(0, sigma, (L, L)); N = (N + N.T) / np.sqrt(2); np.fill_diagonal(N, 0)
    print(f"distances with σ = {sigma} Å error : {100 * triangle_violations(np.abs(D_true + N)):.3f} % of triples violate it")

🔍 **What just happened?** A real structure never violates the triangle inequality; a distance map predicted pair by pair
with 2–4 Å of independent error violates it for 4–8 % of the roughly 300,000 triples — thousands of geometrically
impossible configurations. AlphaFold's triangle updates were designed so that the pair representation can
be made consistent over triples, and its structure module then has to produce actual coordinates, which are consistent
by construction.

**4 · The structure module** (8 blocks with shared weights) represents every residue as a rigid **frame** — a position
and an orientation — and updates them with **invariant point attention** (IPA): attention whose scores depend only on
relative positions of points expressed in residue frames, so that rotating or translating the whole protein changes
nothing but the frames themselves. This is the equivariance of [Module 25](25_graph_neural_networks.ipynb), Section 8.
The module starts from all residues at the origin ("black hole initialisation") and finishes with all backbone and
side-chain atoms.

**5 · Recycling.** The whole network is run three more times, each time receiving its previous outputs — a learned version
of Section 4's lesson that a good starting point matters.

**6 · Training losses and confidence.** The main loss is **FAPE** (frame aligned point error): for every residue frame,
align the predicted and true structures *on that frame* and measure the error of all atoms (clamped at 10 Å). Unlike
RMSD it needs no global superposition, and unlike distance-based scores it is sensitive to chirality. Auxiliary heads
predict distograms, masked MSA residues and the confidences **pLDDT** (per residue) and **PAE** (per pair). PAE is
exactly the computation below: align on residue $i$'s local frame, measure the error of residue $j$.

In [ ]:
# ── 7.3 An aligned-error matrix, as PAE predicts it: the hinge model of Section 3 ──
def aligned_error_matrix(P, Q):
    """E[i, j] = error of residue j after superposing model P on reference Q using only residues i − 1, i, i + 1."""
    n = len(Q)
    E = np.zeros((n, n))
    for i in range(n):
        frame = np.arange(max(0, i - 1), min(n, i + 2))
        if len(frame) < 3:
            frame = np.arange(0, 3) if i == 0 else np.arange(n - 3, n)
        R, t = kabsch(P[frame], Q[frame])
        E[i] = np.linalg.norm(P @ R.T + t - Q, axis=1)
    return E


pae_true = aligned_error_matrix(hinge_models[30], X_true)
pae_dca = aligned_error_matrix(best_models["+ DCA top L/2 (300 seqs)"], X_true)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, M, title in [(axes[0], pae_true, "hinge model (helix 4 swung by 30°)"), (axes[1], pae_dca, "model from 300-sequence DCA")]:
    im = ax.imshow(np.minimum(M, 30), cmap="Greens_r", vmin=0, vmax=30)
    ax.set_xlabel("residue j (error measured)"); ax.set_ylabel("residue i (aligned on)"); ax.set_title(title, fontsize=9)
plt.colorbar(im, ax=axes, shrink=0.8, label="aligned error (Å, capped at 30)")
plt.show()
fixed, moved = np.arange(0, 51), np.arange(51, L)                    # unmoved residues 0–50 | loop tail + helix 4
print(f"hinge model, mean aligned error: unmoved block seen from itself {pae_true[np.ix_(fixed, fixed)].mean():.2f} Å; "
      f"loop tail + helix 4 seen from the unmoved block {pae_true[np.ix_(fixed, moved)].mean():.1f} Å; "
      f"loop tail + helix 4 seen from itself {pae_true[np.ix_(moved, moved)].mean():.2f} Å")

🔍 **What just happened?** For the hinge model the matrix has a clear **block structure**: aligned on any residue of the
unmoved part (residues 0–50), that part is essentially perfect (0.18 Å on average — only residue 50's frame, which
includes a moved neighbour, is slightly off) and the swung block, loop tail + helix 4, is about 8 Å off on average;
aligned on a residue of the swung block, that block is perfect (0.00 Å) and the rest is off.
"Each domain is right, their relative placement is not" is exactly the pattern a PAE plot shows for a protein with a
flexible linker, and it is why PAE — not pLDDT — is the score to look at before trusting the arrangement of domains or
of chains in a complex. The model from the shallow alignment has large errors almost everywhere, except within helices.

**Beyond AlphaFold 2.**
* **RoseTTAFold** (Baek *et al.* 2021) passes information between three tracks — sequence (1-D), residue pairs (2-D) and
  3-D coordinates — and reached accuracy approaching AlphaFold 2's soon after its release.
* **ESMFold** (Lin *et al.* 2023) replaces the MSA by the embeddings of a large protein language model (ESM-2; the
  masked language modelling of [Module 24](24_attention_transformers.ipynb)): one sequence in, structure out, much faster,
  generally somewhat less accurate than AlphaFold 2 with a deep alignment, and best for proteins the language model
  "understands" well (low perplexity).
* **AlphaFold 3** (Abramson *et al.* 2024) handles complexes of proteins, nucleic acids, small molecules, ions and
  modified residues. Its trunk is a simplified "Pairformer", and the structure module is replaced by a **diffusion
  module** that generates all atom coordinates by iterative denoising — exactly the algorithm of
  [Module 28](28_diffusion_flow_matching.ipynb), conditioned on the trunk's representations.

**What these models do not do well (as of 2025).** They predict one static structure, not motions or the distribution of
conformations; intrinsically disordered regions come out as low-confidence spaghetti (or, for diffusion-based models,
sometimes as plausible-looking but spurious structure); accuracy drops for many complexes, antibody–antigen interfaces
and proteins with shallow alignments; and they are not reliable predictors of how a single point mutation changes a
structure or its stability. Confidence scores help, but do not remove the need for experimental validation.

### 📄 Research connection — The structure-prediction revolution

- **What the field does.** The Critical Assessment of protein Structure Prediction (CASP) has run blind prediction experiments since
    1994. AlphaFold 1 (Senior *et al.* 2020) led CASP13 with predicted distograms and gradient-based folding; AlphaFold 2
    (Jumper *et al.* 2021) reached accuracy competitive with experimental structures for a majority of CASP14 targets
    (Kryshtafovych *et al.* 2021), and RoseTTAFold (Baek *et al.* 2021) followed with an openly released three-track
    network. AlphaFold DB (Varadi *et al.* 2022), since expanded to more than 200 million predictions, made them
    available for most known proteins, and
    AlphaFold 3 (Abramson *et al.* 2024) extended prediction to general biomolecular complexes.
- **What this notebook builds.** The geometric ideas inside these systems at toy scale: distances and contacts as a pair representation,
    the triangle inequality behind triangle updates, coordinates from restraints, chirality, superposition-free
    accuracy (lDDT) and the aligned-error matrix that PAE predicts.
- **Where the frontier is.** Conformational ensembles and dynamics (for example, sampling alternative states by subsampling or
    clustering the input alignment; Wayment-Steele *et al.* 2024), accurate complexes and antibody–antigen interfaces,
    ligands and covalent modifications, open and retrainable re-implementations, predictors that report calibrated
    uncertainty, and the link from structure to function and mutation effects.
- **Read next.** Jumper *et al.* 2021 (Nature); Baek *et al.* 2021 (Science); Abramson *et al.* 2024 (Nature).

---
## 8 · 🧭 From toy to real data

> 📍 **Where we are.** Our protein had 68 ideal residues and a perfect, gap-free family. This section is reading: what
> changes for real proteins (no downloads).

**Data resources.** Experimental structures: the [PDB](https://www.rcsb.org) (more than 200,000 entries; mmCIF files with
coordinates, B-factors and metadata) and its regional mirrors. Predicted structures:
[AlphaFold DB](https://alphafold.ebi.ac.uk) (over 200 million predictions with per-residue pLDDT and PAE) and the ESM
Metagenomic Atlas. Classification: [CATH](https://www.cathdb.info) and SCOP. Blind benchmarks:
[CASP](https://predictioncenter.org) (every two years) and [CAMEO](https://www.cameo3d.org) (continuous, weekly).

**Production tools.** [ColabFold](https://github.com/sokrypton/ColabFold) (AlphaFold 2 with fast MMseqs2 alignments, runs
in a GPU notebook; Mirdita *et al.* 2022), the AlphaFold 3 server and code, ESMFold, RoseTTAFold; for comparison and search,
TM-score/TM-align, lDDT (in OpenStructure) and [Foldseek](https://search.foldseek.com); for viewing, PyMOL, ChimeraX and
the Mol* viewer built into the PDB and AlphaFold DB web pages; for scripting, Biopython's `Bio.PDB` and gemmi. A
single-chain prediction takes seconds to minutes on a GPU (longer with alignment search); our toy needed seconds on a CPU.

**The first three things that go wrong.**

1. **The alignment.** Too few effective sequences, or an alignment dominated by close homologues, and every method
   degrades (Section 5). For complexes, pairing the right sequences of the partners is itself a research problem.
2. **Over-reading the model.** Low-pLDDT regions are not structure; high pLDDT for each domain says nothing about their
   arrangement (check PAE); a confident model is still one static snapshot without ligands, partners or membranes unless
   they were modelled.
3. **Unfair comparisons.** Benchmarks leak if test proteins have close homologues in the training set (cluster by
   sequence or structure before splitting, [Module 42](42_trustworthy_ml.ipynb)); scores depend on definitions (Cα or all
   atoms, which residues, which superposition); and an RMSD of a few Å can hide a perfect domain plus a flexible tail.

✏️ **Your turn.** Look up a protein you care about in AlphaFold DB. Which regions have pLDDT below 50? Does the PAE plot
show separate domains? Is there an experimental structure in the PDB, and which of the three scores of Section 3 would
you use to compare them?

---
## 📌 Key takeaways

- A protein's **Cα trace** is described rotation-free by virtual bonds (3.8 Å), virtual angles (≈ 90° in helices) and
  **signed pseudo-torsions** (≈ +50° in right-handed α-helices). The mirror image has identical distances and contact
  map but torsions of −50°; checking torsions validates any structure — measured, predicted or simulated.
- **Superposition** by the **Kabsch algorithm** (SVD of the cross-covariance, with a sign fix against reflections) must
  precede RMSD. **RMSD** is dominated by the worst regions (over 9 Å for a correct protein with one swung helix);
  **TM-score** (0.76) and **lDDT** (0.69, superposition-free) measure how much of the structure is right.
- **Classical MDS** recovers coordinates from a distance matrix (exactly three non-zero eigenvalues) **up to a mirror
  image**; helix handedness resolves it. MDS tolerates noise; incomplete distances need shortest paths and optimisation,
  and **initialisation decides** whether gradient descent finds the fold.
- **Contact-guided folding** from simulated coevolution recovered the fold when the contacts were precise (TM 0.91 with
  all true contacts, 0.79 with DCA's top L/2 from 1,000 sequences) and lost it with false contacts or a shallow alignment
  (TM 0.36): precision beats quantity, alignment depth limits everything, and the mirror image must be resolved.
- An **ensemble's self-consistency** is a useful confidence score: it tracked the true accuracy of whole models closely
  and of individual residues moderately — confidence measures agreement, not truth.
- **AlphaFold 2** = MSA + pair representations, Evoformer (attention, outer product mean, **triangle updates** for
  geometric consistency), a structure module with **invariant point attention** on residue frames, recycling, the FAPE
  loss and confidence heads (pLDDT, **PAE**). AlphaFold 3 swaps the structure module for **diffusion**; ESMFold swaps the
  MSA for a **protein language model**.

---
## ✅ Checkpoint

1. Why must two structures be superposed before computing their RMSD, and why does lDDT not need superposition?
   <details><summary>Answer</summary>Coordinates depend on the arbitrary position and orientation of each structure; RMSD compares coordinates, so the rigid motion must first be removed (Kabsch). lDDT compares internal distances $d_{ij}$, which are unchanged by rotations and translations, so it is superposition-free.</details>
2. What does the SVD step of the Kabsch algorithm do, and why is the factor $d=\operatorname{sign}\det(VU^\top)$ needed?
   <details><summary>Answer</summary>Maximising $\mathrm{tr}(RH)$ over orthogonal $R$ is solved by aligning the singular vectors of the cross-covariance $H=USV^\top$: $R=VU^\top$. If $\det(VU^\top)=-1$ this is a reflection, which would overlay a structure onto its mirror image; flipping the axis with the smallest singular value gives the best *proper* rotation.</details>
3. Why does classical MDS return either the structure or its mirror image, and how did we decide which?
   <details><summary>Answer</summary>Distances — and hence the double-centred matrix $B$ — are identical for a structure and its reflection; the eigenvectors' signs are arbitrary. We computed the helix pseudo-torsions (signed, so they flip under reflection) and reflected the solution if its helices were left-handed.</details>
4. By hand: two structures of $L=68$ residues agree perfectly except for 17 residues that are 20 Å away from their partners after the best superposition. Estimate the TM-score ($d_0\approx2.9$ Å).
   <details><summary>Answer</summary>The 51 correct residues contribute ≈ 1 each, the 17 wrong ones $1/(1+(20/2.9)^2)\approx0.02$ each: TM ≈ (51 + 0.35)/68 ≈ 0.76 — like the 90° hinge model of Section 3 (TM 0.76), whose RMSD was over 9 Å.</details>
5. In Section 5, why did adding more DCA contacts (top L instead of top L/2) make the model worse?
   <details><summary>Answer</summary>The protein has only 38 long-range contacts, so a list of 68 contains at least 30 false ones. Each false contact is an incorrect restraint pulling two distant residues together; enough of them distort the fold (TM fell from 0.79 to 0.36). Precision matters more than the number of restraints.</details>
6. What does AlphaFold's pLDDT measure, and what does PAE add?
   <details><summary>Answer</summary>pLDDT is the network's prediction of the per-residue lDDT-Cα of its model (0–100): local accuracy. PAE predicts, for each pair (i, j), the error of residue j's position when the model is aligned on residue i's frame — so it reveals whether domains or chains are placed correctly relative to each other, which pLDDT cannot.</details>
7. Code reading: in `fold`, what is the role of the term `w_repel * relu(8 − D)² * other`, and what would happen without the clash term?
   <details><summary>Answer</summary>It gently pushes apart pairs that are *not* predicted to be in contact, so the model does not collapse into an overly compact blob that trivially satisfies all contacts. Without the clash term ($d\ge4$ Å) nothing would stop atoms from overlapping, and contacts could be satisfied by unphysical, interpenetrating chains.</details>
8. What problem do AlphaFold's triangle updates address? Use Section 7.2's numbers.
   <details><summary>Answer</summary>Pairwise predictions can be geometrically inconsistent: noisy distance maps violated the triangle inequality for a fraction of triples, which no real structure does. Triangle updates let the representation of pair (i, j) be informed by all pairs (i, k) and (k, j), encouraging consistency over triples before coordinates are built.</details>

## 🧠 Questions & discussion

1. AlphaFold DB contains confident predictions for most human proteins. Which biological questions does a predicted structure answer well, and which still need an experiment (think of dynamics, ligands, complexes, mutations)?
2. Our ensemble confidence tracked accuracy well for whole models but only moderately per residue. Why might a learned confidence head such as pLDDT do better — and in which situations could it be worse than an ensemble?
3. Contact-guided folding failed for a shallow alignment. Many medically important proteins (viral proteins, antibodies, de novo designs) have few homologues. What options do current methods offer for them, and what are their risks?
4. A simulator that drew some helices left-handed would still produce an almost identical contact map, and contact prediction would never notice (an early version of this course's simulator did this; only the torsion check of Section 2 caught it). What does this suggest about validating simulators, benchmarks and training data in computational biology?
5. Structure predictors and generative design tools (Module 39) are freely available. Who benefits, who might be left out (compute, data, expertise), and what responsibilities come with releasing such models?

## 🚀 Bonus — going further

- **β-hairpins.** Write a small generator (modelled on `simulate_helix_bundle`) for a two-stranded β-hairpin (virtual angle ≈ 120°, pseudo-torsion ≈ −170°, strands 4.5–5 Å apart). How do its contact map and pseudo-torsions differ from a helix, and can the folding of Section 5 recover it from contacts alone?
- **Contacts from attention.** Train the small transformer of [Module 24](24_attention_transformers.ipynb) on the family of Section 5, extract APC-corrected attention maps as contact scores, and fold with them. Compare with DCA.
- **Full backbones and Ramachandran.** Build N, Cα and C atoms for an ideal helix and strand (bond lengths 1.46, 1.52, 1.33 Å; standard angles), compute φ and ψ, and place them on a Ramachandran plot.
- **Robust and probabilistic restraints.** Replace binary contacts by a distogram (a probability over distance bins for each pair, e.g. from DCA scores through a logistic link) and fold by minimising the negative log-probability, as AlphaFold 1 did.
- **Try the real thing.** In a GPU Colab, run ColabFold on a small protein with an experimental structure in the PDB, parse both with Biopython, and compute RMSD, TM-score and lDDT with this notebook's functions. Compare your lDDT with the model's pLDDT.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 38 · Protein structure: geometry, contacts and structure prediction
Biological question: how is a protein structure described and compared, how can contacts predicted from a family
                     become 3-D coordinates, and how confident should we be in a predicted model?
Data (simulated): simulate_helix_bundle Cα trace (68 residues, right-handed helices), true distances
                  and contacts; simulate_protein_family (1,000 sequences, Potts couplings on the true contacts)
Method: Cα geometry (bonds, angles, pseudo-torsions); Kabsch superposition; RMSD, TM-score, lDDT; classical MDS;
        stress embedding; mean-field DCA + secondary-structure restraints + gradient-based folding; bootstrap ensembles
Key equation: R = V diag(1, 1, sign det(V Uᵀ)) Uᵀ with H = Σ p_i q_iᵀ = U S Vᵀ;  B = −½ J D⁽²⁾ J = X Xᵀ;
              TM = max (1/L) Σ 1 / (1 + (d_i/d0)²)
Key parameters: contact cut-off (8 Å) and separation (≥ 6), number of contacts (L/2, L), alignment depth,
                restraint weights, number of starts, bootstrap fraction
Ground-truth check result: (helix torsions, protein vs mirror image; Kabsch rotation error; RMSD/TM/lDDT for hinge;
                           MDS eigenvalues and mirror; TM-score per restraint set; confidence vs true lDDT)
What changed when I modified … (number of contacts / alignment depth / starts / bootstrap fraction): …
One failure mode: (e.g. false contacts in the top-L list; mirror image without the handedness check; random starts stuck)
Connection to research (paper/tool): EVfold (Marks et al. 2011); AlphaFold 2 (Jumper et al. 2021); TM-score, lDDT
```

## 📚 References

- Anfinsen C. B. (1973) Principles that govern the folding of protein chains. *Science* 181:223–230.
- Pauling L., Corey R. B., Branson H. R. (1951) The structure of proteins: two hydrogen-bonded helical configurations of the polypeptide chain. *Proc Natl Acad Sci USA* 37:205–211.
- Ramachandran G. N., Ramakrishnan C., Sasisekharan V. (1963) Stereochemistry of polypeptide chain configurations. *J Mol Biol* 7:95–99.
- Berman H. M. *et al.* (2000) The Protein Data Bank. *Nucleic Acids Res* 28:235–242.
- Kabsch W. (1976) A solution for the best rotation to relate two sets of vectors. *Acta Crystallogr A* 32:922–923.
- Zhang Y., Skolnick J. (2004) Scoring function for automated assessment of protein structure template quality. *Proteins* 57:702–710.
- Zhang Y., Skolnick J. (2005) TM-align: a protein structure alignment algorithm based on the TM-score. *Nucleic Acids Res* 33:2302–2309.
- Xu J., Zhang Y. (2010) How significant is a protein structure similarity with TM-score = 0.5? *Bioinformatics* 26:889–895.
- Zemla A. (2003) LGA: a method for finding 3D similarities in protein structures. *Nucleic Acids Res* 31:3370–3374.
- Mariani V., Biasini M., Barbato A., Schwede T. (2013) lDDT: a local superposition-free score for comparing protein structures and models using distance difference tests. *Bioinformatics* 29:2722–2728.
- Basu S., Wallner B. (2016) DockQ: a quality measure for protein-protein docking models. *PLoS ONE* 11:e0161879.
- Torgerson W. S. (1952) Multidimensional scaling: I. Theory and method. *Psychometrika* 17:401–419.
- Morcos F. *et al.* (2011) Direct-coupling analysis of residue coevolution captures native contacts across many protein families. *Proc Natl Acad Sci USA* 108:E1293–E1301.
- Marks D. S. *et al.* (2011) Protein 3D structure computed from evolutionary sequence variation. *PLoS ONE* 6:e28766.
- Senior A. W. *et al.* (2020) Improved protein structure prediction using potentials from deep learning. *Nature* 577:706–710.
- Yang J. *et al.* (2020) Improved protein structure prediction using predicted interresidue orientations. *Proc Natl Acad Sci USA* 117:1496–1503.
- Jumper J. *et al.* (2021) Highly accurate protein structure prediction with AlphaFold. *Nature* 596:583–589.
- Tunyasuvunakool K. *et al.* (2021) Highly accurate protein structure prediction for the human proteome. *Nature* 596:590–596.
- Baek M. *et al.* (2021) Accurate prediction of protein structures and interactions using a three-track neural network. *Science* 373:871–876.
- Kryshtafovych A., Schwede T., Topf M., Fidelis K., Moult J. (2021) Critical assessment of methods of protein structure prediction (CASP) — Round XIV. *Proteins* 89:1607–1617.
- Varadi M. *et al.* (2022) AlphaFold Protein Structure Database: massively expanding the structural coverage of protein-sequence space with high-accuracy models. *Nucleic Acids Res* 50:D439–D444.
- Mirdita M. *et al.* (2022) ColabFold: making protein folding accessible to all. *Nat Methods* 19:679–682.
- Lin Z. *et al.* (2023) Evolutionary-scale prediction of atomic-level protein structure with a language model. *Science* 379:1123–1130.
- Abramson J. *et al.* (2024) Accurate structure prediction of biomolecular interactions with AlphaFold 3. *Nature* 630:493–500.
- van Kempen M. *et al.* (2024) Fast and accurate protein structure search with Foldseek. *Nat Biotechnol* 42:243–246.
- Wayment-Steele H. K. *et al.* (2024) Predicting multiple conformations via sequence clustering and AlphaFold2. *Nature* 625:832–839.

## ➡️ Next

Prediction asks "which structure does this sequence fold into?"; design asks the inverse. **[Module 39 · Generative
design of proteins and molecules](39_generative_design.ipynb)** samples new sequences from a family model, optimises
them with learned surrogates, designs sequences for a given backbone and discusses the RFdiffusion → ProteinMPNN →
AlphaFold pipeline, in which designs are kept only if the predicted and intended structures agree — the comparison
problem of Section 3.